## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" \
pip install llama-cpp-python==0.2.75 --no-cache-dir
# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

In [ ]:
!pip install unidecode
!pip install faiss-cpu

**Note**:
- After running the above cells, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

# LangChain specific imports
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.document_loaders import TextLoader, PyMuPDFLoader
from langchain_community.llms import LlamaCpp

# For downloading models from Hugging Face Hub
from huggingface_hub import hf_hub_download
from langchain.embeddings import HuggingFaceEmbeddings

# For direct llama.cpp interactions if needed
from llama_cpp import Llama

In [ ]:
# to visualise data
import matplotlib.pyplot as plt
import seaborn as sns

# to use regular expressions for manipulating text data
import re

# to manipulate string data
import string

# to remove the accented characters
import unidecode

# to load the natural language toolkit
import nltk
nltk.download('stopwords')    # loading the stopwords
# nltk.download('punkt')    # loading the punkt module used in tokenization
# nltk.download('omw-1.4')    # dependency for tokenization
nltk.download('wordnet')    # loading the wordnet module that is used in stemming

# to remove common stop words
from nltk.corpus import stopwords

# to stem words
from nltk.stem.porter import PorterStemmer

## Question Answering using LLM

### Downloading and Loading the model

In [3]:
# Define the model repository and the file name from HuggingFace Hub
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

In [4]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   45G  192G  19% /
tmpfs            64M     0   64M   0% /dev
shm              25G     0   25G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       242G   50G  193G  21% /opt/bin/.nvidia
tmpfs            26G  3.6M   26G   1% /var/colab
tmpfs            26G     0   26G   0% /proc/acpi
tmpfs            26G     0   26G   0% /proc/scsi
tmpfs            26G     0   26G   0% /sys/firmware


In [ ]:
# Download the model file from HuggingFace Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename,
    force_download=True
)

In [6]:
import os

print("Model path:", model_path)
print("File exists:", os.path.exists(model_path))
print("File size (GB):", os.path.getsize(model_path) / 1e9)

Model path: /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf
File exists: True
File size (GB): 9.229924224


In [7]:
# Initialize the LlamaCpp model with configuration
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.01,
    top_p=0.95,
    #verbose=True,
    verbose=False,
    n_ctx=4096,
    n_batch=1024,
    n_gpu_layers=30, # Adjusted for a smaller model to utilize GPU
)


### Response Functions

In [8]:
def response(query,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    model_output = llm.invoke(
      input=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    # The LlamaCpp model from langchain_community directly returns the text as a string
    return model_output

In [9]:
def response_combinations(query, combination_number=0):
  if combination_number==0:
    max_tokens, temperature, top_p, top_k = 1024, 0, 0.8, 25
  elif combination_number==1:
    max_tokens, temperature, top_p, top_k = 1024, 0.2, 0.85, 35
  elif combination_number==2:
    max_tokens, temperature, top_p, top_k = 1024, 0.2, 0.9, 25
  elif combination_number == 3:
    max_tokens, temperature, top_p, top_k = 1024, 0, 0.9, 50
  else:
    max_tokens, temperature, top_p, top_k = 1024, 0.2, 0.9, 50

  model_output = response(query, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k)
  return f"Max tokens: {max_tokens}, Temperature: {temperature}, top_p: {top_p}, top_k: {top_k} \n\n{model_output}\n{'-'*30}"

In [10]:
def print_all_response_combinations(query, concise_as_well=True):
  query_concise = query + " Be concise without too many filler words."
  for combination_number in range(0, 6):
    # Note: having the model repeat the query with the last set of parameters
    # twice is intentional to observe randomness in the response
    print("Original query")
    print(response_combinations(query, combination_number))
    if concise_as_well:
      print("Concise Query")
      print(response_combinations(query_concise, combination_number))
    print(f'{'-' * 45} \n {'-' * 45}\n')

### Defining queries

In [11]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

### Queries and answers

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [12]:
print(response(query_1))


Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause of death in hospitals. The management of sepsis in a critical care unit (CCU) requires a systematic approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a general protocol for managing sepsis in a CCU:

1. Early recognition:
	* Use a standardized screening tool, such as the Sepsis-3 definition, to identify patients who are at risk of developing sepsis.
	* Monitor patients for signs and symptoms of sepsis, such as fever, tachycardia, tachypnea, and altered mental status.
2. Initial assessment:
	* Perform a thorough physical examination to identify any signs of infection or organ dysfunction.
	* Order blood cultures and other diagnostic tests (such as complete blood count, electrolytes, and renal function tests) to confirm the diagnosis of sepsis.
3. Resuscitation:
	* Administer broad-spectrum antibiotics within the first hour of recognition of sepsis

I note that I had to make max_tokens 1024 for this response to be complete. The responses had a hard cut-off at non-sensible points when I had max_tokens=128 and 512. I also note that some lines are very long when they could be broken up into multiple smaller lines.

However, the response is still thorough and great quality overall

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [13]:
print(response(query_2))



Answer:

Appendicitis is a medical emergency that requires prompt treatment. The common symptoms of appendicitis include:

1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.
2. Nausea and vomiting.
3. Loss of appetite.
4. Fever.
5. Abdominal tenderness and guarding (muscle tension).
6. Abdominal swelling.
7. Diarrhea or constipation.

If you suspect that you or someone else has appendicitis, it is essential to seek medical attention immediately. Appendicitis cannot be cured via medicine; instead, surgery is required to remove the inflamed appendix. The surgical procedure most commonly used to treat appendicitis is an appendectomy, which involves removing the inflamed appendix through a small incision in the abdomen.

There are two types of appendectomies: open and laparoscopic. Open appendectomy involves a larger incision in the abdomen, while laparoscopic appendectomy involves several small incisions and the 

I note that there is the "Answer:" at the start of the response and that is unnecessary. Also, "The common" is not necessarily sound language/grammar, since those are not the ONLY common symptoms of appendicitis

However, this response is still great overall, and is much more concise than the response to Query 1

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [14]:
print(response(query_3))


Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here are some effective treatments and solutions for addressing this condition:

1. Corticosteroid injections: These injections can help suppress the immune system and promote hair growth. They are usually given every 4-6 weeks and can be effective in promoting hair regrowth.
2. Topical corticosteroids: Over-the-counter or prescription topical corticosteroids can be applied directly to the affected area to reduce inflammation and promote hair growth.
3. Minoxidil (Rogaine): This is a solution that is applied to the scalp to stimulate hair growth. It is available over-the-counter and can be effective in promoting hair regrowth.
4. Anthralin: This is a medication that is applied to the skin to reduce inflammation and promote hair growth. It is often used in combination with corticosteroids.
5. Phototherapy: Exposure to specific wavelengths of light, such as ultraviolet B (UVB) or narrowband UV

I like how this response put the answers to each of the questions in the same order as the questions were asked (namely the treatments and then the causes)
I also like how this response is thorough with lots of information, but it could be more concise if the user prefers that

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [15]:
print(response(query_4))


There are several treatments that may be recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The specific treatment plan will depend on the location and severity of the injury, as well as the individual's overall health and medical history. Some common treatments for brain injuries include:

1. Medications: To manage symptoms such as pain, inflammation, and anxiety.
2. Rehabilitation therapy: To help regain lost function and improve cognitive, physical, and emotional abilities. This may include physical therapy, occupational therapy, speech therapy, and cognitive therapy.
3. Surgery: To repair damaged brain tissue or relieve pressure on the brain.
4. Neurostimulation therapies: Such as transcranial magnetic stimulation (TMS) or transcranial direct current stimulation (tDCS), which can help improve cognitive function and reduce symptoms of depression.
5. Cognitive rehabilitation: To help the ind

I like how there were separators (1, 2, ...) for each treatment and how they started off with concise descriptions of each treatment (ex: "Medications", "Surgery", etc.)

However, I feel that some of these treatments are meant for healing the brain tissue while others are about dealing with issues after the fact, and I feel the response should somehow separate the treatments listed based on this distinction

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [16]:
print(response(query_5))



A person who has fractured their leg during a hiking trip requires prompt medical attention to ensure proper healing and prevent complications. Here are some necessary precautions and treatment steps:

1. Stop all activity: The first step is to stop all activity and rest the affected leg immediately. This will help prevent further damage and reduce the risk of complications.
2. Assess the severity of the injury: Evaluate the extent of the fracture and determine if there are any other injuries that require attention.
3. Immobilize the leg: Use a splint or cast to immobilize the affected leg and prevent further movement. This will help reduce pain, swelling, and promote healing.
4. Manage pain: Administer pain medication as prescribed by a healthcare professional. Nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen or naproxen can help manage pain and reduce inflammation.
5. Monitor for signs of infection: Keep an eye out for signs of infection, such as redness, swelling, i

I love this response for how it separated precautions / treatment steps and considerations for the recovery process into two different parts of the response. However, I feel that the response could be more clear how certain precautions from the first category could be applied to the second category as well

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [17]:
print_all_response_combinations(query_1)

Original query
Max tokens: 1024, Temperature: 0, top_p: 0.8, top_k: 25 


Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause of death in hospitals. The management of sepsis in a critical care unit (CCU) requires a systematic approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a general protocol for managing sepsis in a CCU:

1. Early recognition:
	* Use a standardized screening tool, such as the Sepsis-3 definition, to identify patients who are at risk of developing sepsis.
	* Monitor patients for signs and symptoms of sepsis, such as fever, tachycardia, tachypnea, and altered mental status.
2. Initial assessment:
	* Perform a thorough physical examination to identify any signs of infection or organ dysfunction.
	* Order blood cultures and other diagnostic tests (such as complete blood count, electrolytes, and renal function tests) to confirm the diagnosis of sepsis.
3. Resuscitation:
	* Administer 

Observations:

> With temperature=0, the original query goes into great detail, while the query asking for a concise response (which I will refer to as the "concise query") gets a much more concise response, going into less detail and getting most important information but skipping some important information as well. The concise query is also responded to much more quickly by the LLM.

> With temperature=0, the original query goes into great detail, while the query asking for a concise response (which I will refer to as the "concise query") gets a much more concise response, going into less detail and getting most important information but skipping some important information as well. The concise query is also responded to much more quickly by the LLM.
>>For example, when using the first combination of LLM parameters, the concise query omits the part about providing Oxygen therapy to maintain SpO2 ≥ 94%
>> For another example, the original query response names precise tests to use while the concise query response doesn't go into specific tests to use

> When using temperature=0.2, the concise query response is often not concise at all, going into lots of sections with excess delimiters between each section and plenty of times cuts off when reaching the token limit, even though the original query response does not reach the token limit

> The temperature being set to 0.2 also creates a noticeably different response on different iterations of the LLM's response

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [13]:
print_all_response_combinations(query_2)

Original query
Max tokens: 1024, Temperature: 0, top_p: 0.8, top_k: 25 



Answer:

Appendicitis is a medical emergency that requires prompt treatment. The common symptoms of appendicitis include:

1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.
2. Nausea and vomiting.
3. Loss of appetite.
4. Fever.
5. Abdominal tenderness and guarding (muscle tension).
6. Abdominal swelling.
7. Diarrhea or constipation.

If you suspect that you or someone else has appendicitis, it is essential to seek medical attention immediately. Appendicitis cannot be cured via medicine; instead, surgery is required to remove the inflamed appendix.

The surgical procedure used to treat appendicitis is called an appendectomy. There are two types of appendectomies: open and laparoscopic.

1. Open Appendectomy: This is the traditional method, where a single incision is made in the lower right abdomen to remove the inflamed appendix.
2. Lapar

> In this case, both the original and concise query responses are very short, with the concise query responses sometimes being shorter or sometimes being longer than the original query responses.

> Some of the responses say that antibiotics can be used to treat mild cases of appendicitis while other responses say that antibiotics cannot treat appendicitis

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [14]:
print_all_response_combinations(query_3)

Original query
Max tokens: 1024, Temperature: 0, top_p: 0.8, top_k: 25 


Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here are some effective treatments and solutions for addressing this condition:

1. Corticosteroid injections: These injections can help suppress the immune system and promote hair growth. They are usually given every 4-6 weeks and can be effective in promoting hair regrowth.
2. Topical corticosteroids: Over-the-counter or prescription topical corticosteroids can be applied directly to the affected area to reduce inflammation and promote hair growth.
3. Minoxidil (Rogaine): This is a solution that is applied to the scalp to stimulate hair growth. It is available over-the-counter and can be effective in promoting hair regrowth.
4. Anthralin: This is a medication that is applied to the skin to reduce inflammation and promote hair growth. It is often used in combination with corticosteroids.
5. Phototherapy: Exposure to sp

> The concise query responses are great for people who need quick refreshers or are looking for general advice, while it is not at all good for doctors who might need much more detail to reference when treating a patient, and our primary focus is on the latter. On the other hand, the original query response seems satisfactory for medical providers who would need this great level of detail without being too overwhelming.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [15]:
print_all_response_combinations(query_4)

Original query
Max tokens: 1024, Temperature: 0, top_p: 0.8, top_k: 25 


There are several treatments that may be recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The specific treatment plan will depend on the location and severity of the injury, as well as the individual's overall health and medical history. Some common treatments for brain injuries include:

1. Medications: To manage symptoms such as pain, inflammation, and anxiety.
2. Rehabilitation therapy: To help regain lost function and improve cognitive, physical, and emotional abilities. This may include physical therapy, occupational therapy, speech therapy, and cognitive therapy.
3. Surgery: To repair damaged brain tissue or relieve pressure on the brain.
4. Neurostimulation therapies: Such as transcranial magnetic stimulation (TMS) or transcranial direct current stimulation (tDCS), which can help improve cognitive function and re

For the first set of parameters, the concise query response does indeed fulfill our purpose of not having so many filler words while still giving the most relevant information. The original query response does go into more detail about the treatments, though

The original query responses go into proper detail without being too overwhelming

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [16]:
print_all_response_combinations(query_5)

Original query
Max tokens: 1024, Temperature: 0, top_p: 0.8, top_k: 25 



A person who has fractured their leg during a hiking trip requires prompt medical attention to ensure proper healing and prevent complications. Here are some necessary precautions and treatment steps:

1. Stop all activity: The first step is to stop all activity and rest the affected leg immediately. This will help prevent further damage and reduce the risk of complications.
2. Seek medical attention: If the fracture is severe or if there are any signs of nerve or blood vessel damage, seek immediate medical attention. A healthcare professional will assess the severity of the injury and provide appropriate treatment.
3. Immobilize the leg: To prevent further damage and promote healing, the affected leg should be immobilized using a splint or cast. This will help keep the bones in place and reduce pain.
4. Manage pain: Pain management is crucial to ensure the person can rest comfortably and undergo treatment. Medi

The concise response still gets all the relevant information across without lots of filler words

## Data Preparation for RAG

### Loading the Data

** IMPORTANT NOTE: If you load the dataset from Google Drive, it will have a disclaimer on every page ("this file is only for use by [your email]...") and you will have to run additional code (commented out in this notebook) to remove it

In [ ]:
# First, ensure Google Drive is mounted if your file is there.
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path_to_folder = "" # replace with your folder path here
loader = PyMuPDFLoader(path_to_folder+'medical_diagnosis_manual.pdf')
data = loader.load()

In [74]:
if 'data_v1' not in globals():
  data_v1 = data.copy()

data = data_v1.copy()

### Removing Redundant Repeated Parts from the Pages

In [ ]:
for doc in data:
    doc.metadata = {}

# Display the first 5 pages again to confirm metadata is removed
print(data[0], '\n')
print(data[1], '\n')
print(data[2])

Run the following cell ONLY if loading from Google Drive

In [ ]:
'''
data = data[2:]  # Get rid of the first 2 pages with the personal use info

# Define the pattern for the disclaimer
disclaimer_pattern = '' # TODO replace with your disclaimer pattern here

# Iterate through each document and remove the disclaimer from page_content
for doc in data:
    doc.page_content = re.sub(disclaimer_pattern, '', doc.page_content).strip()
'''

In [77]:
print(data[0])

page_content='Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1 - Nutritional Disorders    ...............................................................................................................................................................
53
Chapter 1. Nutrition: General Considerations    ........................................................................................

Here, what we're looking for is the page_content attribute of the data

In [78]:
if 'data_v2' not in globals():
  data_v2 = [d.page_content for d in data]
data = data_v2.copy()

In [79]:
if 'data_v3' not in globals():
  data_v3 = data[14:19].copy() + data[60:].copy()
data = data_v3.copy()

In [80]:
print(data[0])

13 Neurologic Disorders… 1583
14 Pulmonary Disorders… 1823
15 Cardiovascular Disorders… 2015
16 Critical Care Medicine… 2243
17 Genitourinary Disorders… 2303
18 Gynecology and Obstetrics… 2479
19 Pediatrics… 2691
20 Geriatrics… 3069
21 Clinical Pharmacology… 3165
22 Injuries; Poisoning… 3187
23 Special Subjects… 3371
Appendixes… 3489
Ready Reference Guides…3489
Normal Laboratory Values…3491
Trade Names of Some Commonly Used Drugs…3505
Index…3521
Guide For Readers
The Contents (p. vii) shows the pages on which readers can find the Editors and Editorial Board
members, consultants, additional reviewers, and contributors, as well as titles of sections, appendixes,
and the index. Thumb tabs with appropriate abbreviations and section numbers mark each section and
the index.
Each Section begins with its own table of contents, listing chapters and topics in that section. Chapters
are numbered serially from the beginning to the end of the book.
The Index is highly detailed and contains many cro

In [81]:
print(data[4])

PO2
oxygen partial pressure (or tension)
PPD
purified protein derivative (tubercullin)
ppm
parts per million
prn
as needed
PT
prothrombin time
PTT
partial thromboplastin time
q
every (only in dosages)
qid
4 times a day (only in dosages)
RA
rheumatoid arthritis
RBC
red blood cell
RNA
ribonucleic acid
SaO2
arterial oxygen saturation
sc
subcutaneous
sec
second
SI
International System of Units
SIDS
sudden infant death syndrome
SLE
systemic lupus erythematosus
sp
species (when referring to the singular) [eg, Clostridium sp]
spp
species (when referring to the plural) [eg, Nocardia and Myocardia spp]
sp gr
specific gravity
sq
square
SSRI
selective serotonin reuptake inhibitor
TB
tuberculosis
tid
3 times a day (only in dosages)
TNF
tumor necrosis factor
TPN
total parenteral nutrition
TSH
thyroid-stimulating hormone
URI
upper respiratory infection
UTI
urinary tract infection
vs
versus
WBC
white blood cell
WHO
World Health Organization
wk
week
wt
weight
yr
year
μ
micro-; micron
μCi
microcurie
μg

The ending of each page with "The Merck Manual ..." is consistent, repeated, and redundant text that could contaminate our training process

In [82]:
# Remove the ending repeated "The Merck Manual ..." from each page
for idx in range(len(data)):
  page_end_index = data[idx].find('\nThe Merck Manual of Diagnosis & Therapy, 19th Edition')
  data[idx] = data[idx][:page_end_index]

In [83]:
end_index_idx_4 = data[4].find('Editors')
data[4] = data[4][:end_index_idx_4-1]
print(data[4])

PO2
oxygen partial pressure (or tension)
PPD
purified protein derivative (tubercullin)
ppm
parts per million
prn
as needed
PT
prothrombin time
PTT
partial thromboplastin time
q
every (only in dosages)
qid
4 times a day (only in dosages)
RA
rheumatoid arthritis
RBC
red blood cell
RNA
ribonucleic acid
SaO2
arterial oxygen saturation
sc
subcutaneous
sec
second
SI
International System of Units
SIDS
sudden infant death syndrome
SLE
systemic lupus erythematosus
sp
species (when referring to the singular) [eg, Clostridium sp]
spp
species (when referring to the plural) [eg, Nocardia and Myocardia spp]
sp gr
specific gravity
sq
square
SSRI
selective serotonin reuptake inhibitor
TB
tuberculosis
tid
3 times a day (only in dosages)
TNF
tumor necrosis factor
TPN
total parenteral nutrition
TSH
thyroid-stimulating hormone
URI
upper respiratory infection
UTI
urinary tract infection
vs
versus
WBC
white blood cell
WHO
World Health Organization
wk
week
wt
weight
yr
year
μ
micro-; micron
μCi
microcurie
μg

In [84]:
print(data[3])

kg
kilogram
L
liter
lb
pound
LDH
lactic dehydrogenase
M
molar
m
meter
MAOI
monoamine oxidase inhibitor
MCH
mean corpuscular hemoglobin
MCHC
mean corpuscular hemoglobin concentration
mCi
millicurie
MCV
mean corpuscular volume
mEq
milliequivalent
Mg
magnesium
mg
milligram
MI
myocardial infarction
MIC
minimum inhibitory concentration
min
minute
mIU
milli-international unit
mL
milliliter
mm
millimeter
mmol
millimole
mo
month
mOsm
milliosmole
MRI
magnetic resonance imaging
N
nitrogen; normal (strength of solution)
Na
sodium
NaCl
sodium chloride
ng
nanogram (= millimicrogram)
NGT
nasogastric tube
nm
nanometer (= millimicron)
nmol
nanomole
npo
nothing by mouth
NSAID
nonsteroidal anti-inflammatory drug
O2
oxygen
OTC
over-the-counter (pharmaceuticals)
oz
ounce
P
phosphorus
PACO2
alveolar carbon dioxide partial pressure
PaCO2
arterial carbon dioxide partial pressure
PAO2
alveolar oxygen partial pressure
PaO2
arterial oxygen partial pressure
PCO2
carbon dioxide partial pressure (or tension)
PCR
p

In [85]:
print(data[15])

of breastfeeding, which typically occurs when a younger sibling is born, displacing the older child from the
breast. So children with kwashiorkor tend to be older than those with marasmus. Kwashiorkor may also
result from an acute illness, often gastroenteritis or another infection (probably secondary to cytokine
release), in a child who already has PEU. A diet that is more deficient in protein than energy may be more
likely to cause kwashiorkor than marasmus. Less common than marasmus, kwashiorkor tends to be
confined to specific parts of the world, such as rural Africa, the Caribbean, and the Pacific islands. In
these areas, staple foods (eg, yams, cassavas, sweet potatoes, green bananas) are low in protein and
high in carbohydrates. In kwashiorkor, cell membranes leak, causing extravasation of intravascular fluid
and protein, resulting in peripheral edema.
Starvation is a complete lack of nutrients. It occasionally occurs when food is available (as in fasting or
anorexia nervosa) bu

In [86]:
print(data[1524])

The eruption may be generalized (in severe cases) or more limited but almost always involves the upper
trunk. Ulcerated lesions may develop on the mucous membranes, including the oropharynx and upper
respiratory tract, palpebral conjunctiva, and rectal and vaginal mucosa. In the mouth, vesicles rupture
immediately, are indistinguishable from those of herpetic gingivostomatitis, and often cause pain during
swallowing. Scalp lesions may result in tender, enlarged suboccipital and posterior cervical lymph nodes.
New lesions usually cease to appear by the 5th day, and the majority are crusted by the 6th day; most
crusts disappear < 20 days after onset.
Complications: Secondary bacterial infection (typically streptococcal or staphylococcal) of the vesicles
may occur, causing cellulitis or rarely streptococcal toxic shock. Pneumonia may complicate severe
chickenpox in adults, neonates, and immunocompromised patients of all ages but usually not in
immunocompetent young children. Myocarditis, 

In [87]:
if 'data_v4' not in globals():
  data_v4 = data.copy()
data = data_v4.copy()

With this, our data no longer has things like authors, editors, and repeated, redundant text like "\nThe Merck Manuals ..." at the bottom of each page

### Removing special characters, stopwords, lowercasing, stripping whitespace, and stemming

In [88]:
def apply_function_to_data(data, function):
  new_data = [function(page) for page in data]
  return new_data

In [89]:
# defining a function to remove special characters
def remove_special_characters(text):
    # Defining the regex pattern to match non-alphanumeric characters
    pattern = '[^A-Za-z0-9]+'

    # Finding the specified pattern and replacing non-alphanumeric characters with a blank string
    new_text = ''.join(re.sub(pattern, ' ', text))

    return new_text

In [90]:
# defining a function to remove stop words using the NLTK library
def remove_stopwords(text):
    # Split text into separate words
    words = text.split()

    # Removing English language stopwords
    new_text = ' '.join([word for word in words if word not in stopwords.words('english')])

    return new_text

In [91]:
ps = PorterStemmer()
# defining a function to perform stemming
def apply_porter_stemmer(text):
    # Split text into separate words
    words = text.split()

    # Applying the Porter Stemmer on every word of a message and joining the stemmed words back into a single string
    new_text = ' '.join([ps.stem(word) for word in words])

    return new_text

In [92]:
data = apply_function_to_data(data, remove_special_characters)
data = apply_function_to_data(data, str.lower)
data = apply_function_to_data(data, str.strip)
print("done")
data = apply_function_to_data(data, remove_stopwords)
print("done 2")
data = apply_function_to_data(data, apply_porter_stemmer)

done
done 2


In [93]:
print(data[0])

13 neurolog disord 1583 14 pulmonari disord 1823 15 cardiovascular disord 2015 16 critic care medicin 2243 17 genitourinari disord 2303 18 gynecolog obstetr 2479 19 pediatr 2691 20 geriatr 3069 21 clinic pharmacolog 3165 22 injuri poison 3187 23 special subject 3371 appendix 3489 readi refer guid 3489 normal laboratori valu 3491 trade name commonli use drug 3505 index 3521 guid reader content p vii show page reader find editor editori board member consult addit review contributor well titl section appendix index thumb tab appropri abbrevi section number mark section index section begin tabl content list chapter topic section chapter number serial begin end book index highli detail contain mani cross entri addit reader find mani cross refer throughout text specif page addit relat inform found run head carri section number titl left hand page chapter number titl right hand page abbrevi symbol use throughout text essenti space saver list page ix x abbrevi text expand first mention chapter

In [94]:
print(data[1])

drug design text gener nonproprietari name appendix iii mani drug mention book list alphabet gener name follow one trade name section 23 special subject discuss clinic decis make radiolog imag complementari altern medicin dietari supplement genet smoke cessat rehabilit care surgic patient financi issu health care care die patient among other import author review editor book made extens effort ensur treatment drug dosag regimen accur conform standard accept time public howev constant chang inform result continu research clinic experi reason differ opinion among author uniqu aspect individu clinic situat possibl human error prepar extens text requir reader exercis individu judgment make clinic decis necessari consult compar inform sourc particular reader advis check product inform provid manufactur drug product prescrib administ especi drug unfamiliar use infrequ note reader find date inform addit tabl figur well multimedia enhanc www merckmanu com visit web site frequent new enhanc late

This text now looks cleaned the way I would want it to be

In [97]:
if 'data_v5' not in globals():
  data_v5 = data.copy()
data = data_v5.copy()

### Checking for missing or duplicate values

In [98]:
# To check for None values in a list
none_count = sum(1 for item in data if item is None)
print(f"Number of None values: {none_count}")

# To check for empty strings in a list of strings
empty_string_count = sum(1 for item in data if isinstance(item, str) and not item.strip())
print(f"Number of empty strings: {empty_string_count}")

Number of None values: 0
Number of empty strings: 0


In [99]:
data_df = pd.DataFrame(data, columns=['page_content'])
print(data_df.duplicated().sum())

0


No missing, null, or duplicate values, so no modification needed there

In [100]:
del data_df
# Save RAM

### Data Overview

#### Checking the first 5 pages

In [101]:
# Check the first 5 pages of the data
for i in range(0, 5):
  print(data[i])
  print(f'{'-'*30}')

# print(data[0:5])
# I could do this but then it makes the output all appear on 1 line

13 neurolog disord 1583 14 pulmonari disord 1823 15 cardiovascular disord 2015 16 critic care medicin 2243 17 genitourinari disord 2303 18 gynecolog obstetr 2479 19 pediatr 2691 20 geriatr 3069 21 clinic pharmacolog 3165 22 injuri poison 3187 23 special subject 3371 appendix 3489 readi refer guid 3489 normal laboratori valu 3491 trade name commonli use drug 3505 index 3521 guid reader content p vii show page reader find editor editori board member consult addit review contributor well titl section appendix index thumb tab appropri abbrevi section number mark section index section begin tabl content list chapter topic section chapter number serial begin end book index highli detail contain mani cross entri addit reader find mani cross refer throughout text specif page addit relat inform found run head carri section number titl left hand page chapter number titl right hand page abbrevi symbol use throughout text essenti space saver list page ix x abbrevi text expand first mention chapter

#### Checking the number of pages

In [102]:
print(len(data))

4057


Normally it should be 4114 but remember that I removed many redundant pages in the previous section

### Data Chunking

In [103]:
from langchain_core.documents import Document

# Convert list of strings to list of Document objects
documents = [Document(page_content=d) for d in data]

chunk_size_overlaps = [(1000, 200), (1000, 300), (1500, 300), (2000, 400)]
text_splitters = []
chunks_set = []
text_splitter_chunk_embedding_list = []
for i in range(len(chunk_size_overlaps)):
  chunk_size_overlap = chunk_size_overlaps[i]
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size_overlap[0], chunk_overlap=chunk_size_overlap[1])
  chunks = text_splitter.split_documents(documents)
  text_splitters.append(text_splitter)
  chunks_set.append(chunks)

### Embedding

In [104]:
embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

/tmp/ipykernel_5146/1028113912.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")


### Vector Database

In [105]:
vector_databases = []
for i in range(len(text_splitters)):
  text_splitter = text_splitters[i]
  chunks = chunks_set[i]
  vector_databases.append(FAISS.from_documents(chunks, embedding=embeddings))
  print(i)

0
1
2
3


In [106]:
vector_databases[0].index.reconstruct(0)

array([-3.34040448e-02,  6.97075110e-03, -5.74622601e-02, -2.48407703e-02,
        4.86572459e-02,  2.52448544e-02,  6.02398068e-03, -2.02186685e-03,
        1.39499232e-02, -3.50366943e-02,  3.12608592e-02, -6.85067382e-03,
       -7.30841383e-02,  1.85384545e-02,  8.60081613e-03,  6.46442920e-02,
        3.93964797e-02,  3.01572196e-02,  3.21281888e-02, -4.98774052e-02,
        2.89429240e-02, -3.02249379e-02, -2.65742335e-02,  1.16353165e-02,
       -1.39718624e-02,  1.51510620e-02,  3.33279409e-02,  2.77601276e-02,
       -3.05356756e-02, -2.79409159e-02,  2.53471751e-02,  3.70320082e-02,
        2.56521814e-02, -2.73550786e-02,  3.20770480e-02, -4.11664136e-03,
       -7.80890463e-03,  4.43026843e-03,  1.63456965e-02,  8.65828805e-03,
       -6.05423041e-02, -4.81523387e-02, -3.53677757e-02, -1.68641214e-03,
       -4.40666191e-02,  1.42325386e-02, -2.42517553e-02,  1.36689702e-02,
       -4.81253117e-03, -2.24352247e-04, -2.35094279e-02, -2.08927654e-02,
        2.34429631e-02,  

### Retriever

### System and User Prompt Template

In [111]:
# Template for providing evaluation input
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

### Response Function

In [112]:
def generate_rag_response(user_input, vector_db, k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    # This template will be used specifically for generating the answer.
    # It guides the LLM to provide the answer directly based on the question and context.
    generation_prompt_template = """
###Question
{question}
----------------------------
###Context
{context}
--------------------------------
Based on the provided context, please answer the question directly, without listing the references.
Answer:
"""

    # Retrieve relevant document chunks
    retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": k})
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    # Populate the generation prompt template
    prompt_for_llm = generation_prompt_template.replace('{context}', context_for_query)
    prompt_for_llm = prompt_for_llm.replace('{question}', user_input)

    # Generate the response
    try:
        response = llm(
                  prompt=prompt_for_llm,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k,
                  stop=['\n###', '###', 'Question:', 'Context:', 'Answer:', 'INST']
                  )

        # The LlamaCpp model returns the text directly for invoke, or as choices[0].text for call
        # Assuming `llm` is `LlamaCpp` and directly callable as `llm(prompt=...)` from langchain_community
        # which usually returns the raw model output string or a dict if verbose=True, we need to adapt.
        # Given previous examples `response['choices'][0]['text']` for evaluation, let's assume it's a dict output.
        if isinstance(response, dict) and 'choices' in response and len(response['choices']) > 0:
            return response['choices'][0]['text']
        else:
            return str(response) # Return as string if it's not the expected dict format

    except Exception as e:
        return f'Sorry, I encountered the following error: \n {e}'


**Note that, here, the retriever is defined inside the function itself, thus I am using different retrievers**

## Question Answering using RAG with Fine-Tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [113]:
response_1a = generate_rag_response(query_1, vector_databases[0], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1a)


The protocol for managing sepsis in a critical care unit involves early recognition and aggressive treatment of severe sepsis and septic shock. The approach includes:

1. Early diagnosis: Suspect sepsis in any patient with suspected or confirmed infection, especially if they have signs of organ dysfunction or hypoperfusion.
2. Rapid fluid resuscitation: Administer IV fluids to restore mean arterial pressure (MAP) to ≥65 mmHg and serum lactate level to <2 mmol/L within the first 6 hours of recognition.
3. Broad-spectrum antibiotics: Start broad-spectrum antibiotics within the first hour of recognition, and consider adding a second antibiotic if there is no improvement after 48 hours.
4. Vasopressors: Use vasopressors to maintain MAP ≥65 mmHg if fluid resuscitation is not sufficient.
5. Mechanical ventilation: Provide mechanical ventilation as needed, with a goal of maintaining PaO2/FiO2 ratio ≥100 mmHg.
6. Monitoring: Closely monitor the patient's clinical status, laboratory values, an

In [114]:
response_1b = generate_rag_response(query_1, vector_databases[0], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1b)


The protocol for managing sepsis in a critical care unit includes:

1. Early recognition and suspicion of sepsis, including signs such as fever, tachycardia, tachypnea, and confusion.
2. Aggressive fluid resuscitation within the first 6 hours of diagnosis, with administration of normal saline and careful monitoring of central venous pressure (CVP) and pulmonary artery pressure (PAP).
3. Broad-spectrum antibiotics should be administered within the first hour of diagnosis, and the choice of antibiotics should be guided by the suspected source of infection and the patient's allergies and medical history.
4. Management of lactic acidosis and electrolyte imbalances, including administration of sodium bicarbonate and potassium supplements as needed.
5. Close monitoring of vital signs, including temperature, blood pressure, tachycardia, and tachypnea, as well as frequent assessment of organ function, such as renal and hepatic function.
6. Early recognition and management of sepsis-induced ca

In [115]:
response_1c = generate_rag_response(query_1, vector_databases[1], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1c)


The protocol for managing sepsis in a critical care unit should include the following steps:

1. Early recognition and prompt treatment of sepsis, including administration of broad-spectrum antibiotics and source control measures.
2. Aggressive fluid resuscitation to maintain mean arterial pressure (MAP) ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/h.
3. Management of underlying causes of sepsis, such as pneumonia or urinary tract infections.
4. Close monitoring of vital signs, including temperature, blood pressure, tachycardia, and tachypnea.
5. Regular assessment of organ dysfunction, such as acute kidney injury (AKI) and hepatic dysfunction.
6. Use of invasive and non-invasive ventilation strategies as needed.
7. Protocolized management of sepsis-induced cardiovascular dysfunction, including vasopressors and inotropes as needed.
8. Prophylaxis against deep vein thrombosis (DVT) and pulmonary embolism (PE).
9. Early recognition and management of sepsis-associated organ failure, such as ac

In [116]:
response_1d = generate_rag_response(query_1, vector_databases[1], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1d)


The protocol for managing sepsis in a critical care unit includes:

1. Early recognition and prompt treatment of sepsis, including administration of broad-spectrum antibiotics and fluid resuscitation.
2. Aggressive fluid management to maintain mean arterial pressure (MAP) ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/h.
3. Use of vasopressors if necessary to maintain MAP ≥ 65 mmHg.
4. Management of hypothermia with warming devices or warm fluids.
5. Prophylaxis for deep vein thrombosis (DVT) and pulmonary embolism (PE).
6. Early recognition and management of sepsis-induced organ dysfunction, such as acute respiratory distress syndrome (ARDS), acute kidney injury (AKI), and cardiovascular instability.
7. Use of invasive monitoring devices, such as arterial lines and central venous catheters, to monitor vital signs and fluid status.
8. Regular assessment of the patient's clinical status and response to treatment.
9. Early involvement of critical care specialists and other multidisciplinary tea

In [117]:
response_1e = generate_rag_response(query_1, vector_databases[2], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1e)


The protocol for managing sepsis in a critical care unit includes:

1. Early recognition and diagnosis of sepsis, including the use of clinical criteria such as body temperature, heart rate, respiratory rate, and white blood cell count.
2. Aggressive fluid resuscitation to restore blood volume and perfusion pressure.
3. Administration of broad-spectrum antibiotics to treat suspected gram-negative bacterial infections.
4. Surgical drainage of infected tissue or abscesses, if necessary.
5. Close monitoring of vital signs, including temperature, heart rate, respiratory rate, and blood pressure.
6. Regular assessment of organ function, including renal, hepatic, and cardiovascular parameters.
7. Management of hypotension and shock with vasopressors such as dopamine or norepinephrine, as needed.
8. Proactive management of acidosis and electrolyte imbalances.
9. Regular blood cultures to identify the causative organism and guide antibiotic therapy.
10. Early recognition and management of sep

In [118]:
response_1f = generate_rag_response(query_1, vector_databases[2], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1f)

The protocol for managing sepsis in a critical care unit includes:

1. Early recognition and diagnosis of sepsis based on clinical signs and laboratory tests.
2. Aggressive fluid resuscitation with normal saline or lactated Ringer's solution to maintain mean arterial pressure (MAP) ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/h.
3. Broad-spectrum antibiotics within the first hour of recognition of sepsis, and targeted antibiotics thereafter based on the suspected etiology.
4. Management of hypotension with vasopressors such as norepinephrine or dopamine to maintain MAP ≥ 65 mmHg.
5. Close monitoring of vital signs, including temperature, heart rate, respiratory rate, and oxygen saturation.
6. Regular assessment of fluid balance and electrolyte levels.
7. Protocolized management of sepsis-induced organ dysfunction, such as acute respiratory distress syndrome (ARDS), acute kidney injury (AKI), and cardiac dysfunction.
8. Early recognition and management of sepsis-associated coagulopathy and di

In [119]:
response_1g = generate_rag_response(query_1, vector_databases[3], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_1g)

The protocol for managing sepsis in a critical care unit includes:

1. Early recognition and diagnosis of sepsis, including signs and symptoms such as fever, hypotension, tachycardia, and tachypnea.
2. Aggressive fluid resuscitation with crystalloids or colloids to restore intravascular volume and blood pressure.
3. Broad-spectrum antibiotics within the first hour of recognition of sepsis, and administration of antibiotics appropriate for the suspected etiology of sepsis.
4. Management of underlying infections, such as pneumonia or urinary tract infections, with appropriate antibiotics and supportive care.
5. Close monitoring of vital signs, including temperature, blood pressure, tachycardia, and tachypnea, as well as frequent assessment of organ dysfunction and sepsis severity.
6. Early recognition and management of septic shock, including the use of vasopressors to maintain mean arterial pressure ≥65 mmHg and the administration of corticosteroids for patients with severe sepsis or se

I like response 1f most due to having the most specific numbers and still covering all the core details

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [120]:
response_2a = generate_rag_response(query_2, vector_databases[0], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2a)


The common symptoms of appendicitis include:

* Sudden and severe pain in the lower right abdomen that begins around the navel and then moves to the lower right abdomen
* Nausea and vomiting
* Loss of appetite
* Abdominal tenderness and guarding (muscle tension)
* Fever (usually less than 101°F)
* Abdominal swelling and bloating

If the diagnosis is confirmed, surgical intervention is necessary to treat appendicitis. The most common surgical procedure for treating appendicitis is a laparoscopic appendectomy, where a small incision is made in the abdomen, and a laparoscope (a thin tube with a camera and light) is inserted to visualize the appendix. If the appendix has already ruptured, an open appendectomy may be performed instead. In some cases, antibiotics may be given before surgery to help treat any infection.

It's important to note that while appendicitis can be treated with antibiotics, it is not a condition that can be cured solely with medicine. Surgical intervention is necess

In [121]:
response_2b = generate_rag_response(query_2, vector_databases[0], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2b)


The common symptoms of appendicitis include:

* Sudden and severe pain in the lower right abdomen that begins around the navel and then moves to the lower right abdomen
* Nausea and vomiting
* Loss of appetite
* Abdominal tenderness and guarding (muscle tension)
* Fever (usually less than 101°F)
* Abdominal swelling and bloating
* Palpable (able to be felt) tender mass in the lower right abdomen

If the symptoms are severe and there is a high suspicion of appendicitis, surgical intervention is necessary. The surgical procedure for treating appendicitis is an appendectomy, which involves removing the inflamed appendix. In some cases, the appendix may be removed through laparoscopic surgery, which is a minimally invasive procedure that uses a small camera and specialized instruments to remove the appendix. However, if the appendix has ruptured or there are signs of peritonitis (inflammation of the lining of the abdominal cavity), an open appendectomy may be necessary, which involves a l

In [122]:
response_2c = generate_rag_response(query_2, vector_databases[1], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2c)


The common symptoms of appendicitis include:

* Sudden and severe pain in the lower right abdomen that begins around the navel and then moves to the lower right abdomen
* Nausea and vomiting
* Loss of appetite
* Abdominal tenderness and guarding (muscle tension)
* Fever (usually over 100.4°F/38°C)
* Abdominal swelling and bloating
* Palpable (able to be felt) tenderness in the right lower quadrant of the abdomen
* Rigidity (stiffness) in the abdomen
* Cough or motion intolerance due to pain
* Pain that worsens with movement or coughing
* Abdominal guarding (tightening of the abdominal muscles)
* Omentum (a layer of fat and membrane that covers the intestines) tenderness
* Peritonitis (inflammation of the peritoneum, the lining of the abdominal cavity)

If the symptoms are severe and there is a high level of suspicion, surgical intervention is necessary to treat appendicitis. The surgical procedure typically involves laparoscopic or open appendectomy, where the inflamed appendix is rem

In [123]:
response_2d = generate_rag_response(query_2, vector_databases[1], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2d)


The common symptoms of appendicitis include abdominal pain, nausea, vomiting, loss of appetite, fever, and tenderness in the right lower quadrant of the abdomen. However, these symptoms can be non-specific and may resemble other conditions such as a viral gastroenteritis or a urinary tract infection.

If the diagnosis is not clear, imaging studies such as CT scans or ultrasound may be ordered to confirm the presence of an inflamed appendix. In some cases, the diagnosis may be made based on the patient's symptoms and physical examination findings, without the need for imaging studies.

If the diagnosis is confirmed, surgical removal of the inflamed appendix is the standard treatment. The surgical procedure most commonly used is a laparoscopic appendectomy, which involves making several small incisions in the abdomen and using a camera and specialized instruments to remove the inflamed appendix. In some cases, an open appendectomy may be necessary if the appendix has ruptured or if ther

In [125]:
response_2e = generate_rag_response(query_2, vector_databases[2], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2d)


The common symptoms of appendicitis include abdominal pain, nausea, vomiting, loss of appetite, fever, and tenderness in the right lower quadrant of the abdomen. However, these symptoms can be non-specific and may resemble other conditions such as a viral gastroenteritis or a urinary tract infection.

If the diagnosis is not clear, imaging studies such as CT scans or ultrasound may be ordered to confirm the presence of an inflamed appendix. In some cases, the diagnosis may be made based on the patient's symptoms and physical examination findings, without the need for imaging studies.

If the diagnosis is confirmed, surgical removal of the inflamed appendix is the standard treatment. The surgical procedure most commonly used is a laparoscopic appendectomy, which involves making several small incisions in the abdomen and using a camera and specialized instruments to remove the inflamed appendix. In some cases, an open appendectomy may be necessary if the appendix has ruptured or if ther

In [126]:
response_2f = generate_rag_response(query_2, vector_databases[2], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2f)


The common symptoms for appendicitis include abdominal pain, nausea, vomiting, loss of appetite, fever, and abdominal tenderness. The pain typically starts near the navel and then moves to the lower right abdomen. However, in some cases, the pain may be diffuse and not localized.

If the symptoms are severe and there is a high suspicion of appendicitis, surgical intervention is necessary. The surgical procedure for treating appendicitis is usually an appendectomy, which involves removing the inflamed appendix. In some cases, the appendix may be removed through laparoscopic surgery, which is a minimally invasive procedure that uses a small camera and specialized instruments to remove the appendix.

If the appendix is not removed promptly, it can lead to complications such as perforation, abscess formation, and gangrene, which can be life-threatening. Therefore, it is essential to seek medical attention immediately if symptoms of appendicitis are present.


In [127]:
response_2g = generate_rag_response(query_2, vector_databases[3], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_2g)


The common symptoms for appendicitis include:

* Sudden and severe pain in the lower right abdomen that begins around the navel and then moves to the lower right abdomen
* Nausea and vomiting
* Loss of appetite
* Abdominal tenderness and guarding (muscle tension)
* Fever
* Abdominal swelling

If the symptoms are severe, it is important to seek immediate medical attention. Appendicitis can be treated with antibiotics or surgery, depending on the severity of the condition. In cases where the appendix has ruptured, surgery is usually required to remove the inflamed tissue and prevent further complications.

In some cases, the symptoms may be similar to other conditions such as hernia or ectopic pregnancy, so it is important to seek medical attention if there are any concerns about appendicitis. A physical examination, imaging tests such as a CT scan or ultrasound, and laboratory tests can help confirm the diagnosis and determine the appropriate course of treatment.


Again, I feel response 2f is most detailed, but several other responses are almost as detailed

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [128]:
response_3a = generate_rag_response(query_3, vector_databases[0], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3a)

There are several effective treatments and solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp. These include:

1. Minoxidil (2% solution) applied topically to the affected area, which can help stimulate hair growth and slow down hair loss.
2. Topical corticosteroids, such as triamcinolone, which can help reduce inflammation and suppress the immune system's attack on hair follicles.
3. Anthralin (5-fluorouracil) applied topically, which can help reduce inflammation and promote hair growth.
4. Immunotherapy with diphencyprone (DPCP) or squaric acid dibutyl ester (SADBE), which can help stimulate the immune system to attack the underlying cause of alopecia areata.
5. Oral antifungals, such as griseofulvin, which can help treat fungal infections that may be contributing to hair loss.
6. Psychotherapy and behavior modification techniques, such as cognitive-behavioral therapy (CBT), which can help address underlying psychological factors that

In [129]:
response_3b = generate_rag_response(query_3, vector_databases[0], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3b)

There are several effective treatments for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp. These include:

1. Minoxidil (2% solution or foam): Applied topically to the affected area, minoxidil can help stimulate hair growth and slow down hair loss. It is available over-the-counter and is one of the most commonly used treatments for hair loss.
2. Finasteride (1mg daily): This oral medication works by blocking the production of dihydrotestosterone (DHT), a hormone that contributes to hair loss. It is effective in treating male pattern baldness and female pattern hair loss.
3. Topical corticosteroids: These medications can help reduce inflammation and suppress the immune system, which can help slow down hair loss. They are available in various strengths and may be prescribed for short periods to control sudden patchy hair loss.
4. Anthralin (2.5% cream or ointment): This medication is used to reduce inflammation and promote hair growth. It is applie

In [130]:
response_3c = generate_rag_response(query_3, vector_databases[1], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3c)

There are several effective treatments and solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp. These include:

1. Topical corticosteroids: These medications can help reduce inflammation and slow down hair loss. They are available in different strengths, and your doctor may recommend a combination of high- and low-potency corticosteroids.
2. Topical minoxidil: This medication is applied directly to the scalp and can stimulate hair growth by increasing blood flow and widening hair follicles. It is available in different concentrations, and your doctor may recommend a higher concentration for more severe hair loss.
3. Anthralin: This medication is used to treat psoriasis, but it can also help promote hair growth. It works by reducing inflammation and increasing blood flow to the scalp.
4. Immunotherapy: This treatment involves using injections or oral medications to stimulate the immune system and promote hair growth. It is often used for 

In [132]:
response_3d = generate_rag_response(query_3, vector_databases[1], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3d)

There are several effective treatments and solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp. These include:

1. Topical corticosteroids: These medications can help reduce inflammation and slow down hair loss. They are available in different strengths, and your doctor will determine the appropriate one for your condition.
2. Topical minoxidil: This medication stimulates hair growth and can help reverse hair loss. It is applied directly to the affected area and is available over-the-counter.
3. Anthralin: This medication helps reduce inflammation and promote hair growth. It is often used in combination with corticosteroids.
4. Immunotherapy: This treatment option involves using injections or oral medications to stimulate the immune system and promote hair growth.
5. Platelet-rich plasma (PRP) therapy: This is a non-surgical treatment that involves injecting platelet-rich plasma into the affected area to stimulate hair growth.
6. Low-lev

In [133]:
response_3e = generate_rag_response(query_3, vector_databases[2], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3e)


Sudden patchy hair loss, commonly seen as localized bald spots on the scalp, can be caused by various factors such as autoimmune disorders like alopecia areata, thyroid disorders, stress, and infectious diseases like tinea capitis. The possible causes behind it include:

1. Autoimmune disorders: Alopecia areata is an autoimmune condition where the immune system attacks the hair follicles, leading to patchy hair loss.
2. Thyroid disorders: An underactive or overactive thyroid can cause hair loss, especially in women.
3. Stress: Prolonged stress can cause hair loss due to the increased production of cortisol, a hormone that can disrupt the normal hair growth cycle.
4. Infectious diseases: Fungal infections like tinea capitis can cause patchy hair loss, especially in children.

Effective treatments for sudden patchy hair loss include:

1. Topical corticosteroids: These medications can help reduce inflammation and promote hair growth.
2. Minoxidil: This medication can stimulate hair growt

In [134]:
response_3f = generate_rag_response(query_3, vector_databases[2], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3f)


The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and the possible causes behind it include:

1. Topical corticosteroids: These can be applied directly to the affected area to reduce inflammation and promote hair growth.
2. Minoxidil: This medication can be applied topically to stimulate hair growth and slow down hair loss.
3. Anthralin: This medication can be used to treat psoriasis, which is a common cause of patchy hair loss.
4. Immunotherapy: This treatment option involves using injections or oral medications to suppress the immune system and prevent it from attacking the hair follicles.
5. Oral antifungals: These can be used to treat fungal infections that may be contributing to patchy hair loss.
6. Hormone modulators: In cases where hormonal imbalances are suspected to be causing hair loss, medications such as spironolactone and finasteride may be prescribed to regulate hormone levels.
7. Surgical op

In [135]:
response_3g = generate_rag_response(query_3, vector_databases[3], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_3g)


The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, include:

1. Minoxidil (2% solution) for male/female pattern hair loss and alopecia areata.
2. Finasteride (1mg/day) for male pattern hair loss.
3. Topical corticosteroids for short-term use in severe cases of alopecia areata.
4. Anthralin (0.5% cream) for alopecia areata.
5. Immunotherapy with diphencyprone (DPCP) and psoralen plus ultraviolet A (PUVA) for severe alopecia areata.
6. Surgical options, such as follicular transplantation or scalp flap, for more advanced cases of hair loss.
7. Treatment of underlying conditions, such as thyroid disorders or autoimmune diseases, that may be contributing to the hair loss.
8. Reducing stress and using gentle hair care practices to prevent traction alopecia.
9. Using wigs or hairpieces to cover up hair loss due to chemotherapy or other forms of hair loss.
10. Considering oral antifungals, such as griseofulvin, for

Response 3d is best due to giving the highest number of relevant suggestions and including suggestions not present in other responses, though these suggestions would be used in rare cases. Response 3f still focuses on common causes and treatments

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [136]:
response_4a = generate_rag_response(query_4, vector_databases[0], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4a)


Treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, may include:

1. Rapid neurological assessment and evaluation to identify the severity and location of the injury.
2. Immediate management of airway, breathing, and circulation (ABCs) to ensure proper oxygenation and prevent further damage.
3. Control of external bleeding and immobilization of the spine to prevent further injury.
4. Surgical intervention to relieve pressure, evacuate hematomas, and repair any structural damage.
5. Pain management with short-acting opioids such as fentanyl.
6. Rehabilitation services including physical, occupational, and speech therapy to address neurological deficits and improve functional abilities.
7. Counseling and support for the patient and their family to address emotional and social needs.
8. Serial CT scans to monitor for any changes in the brain tissue and to guide further treatment.
9. Early mobilizat

In [137]:
response_4b = generate_rag_response(query_4, vector_databases[0], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4b)


Treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, include:

1. Rapid neurological assessment and evaluation to identify the severity and location of the injury.
2. Immediate management of airway, breathing, and circulation to prevent further damage.
3. Control of external bleeding and immobilization of the spine to prevent further injury.
4. Early surgical intervention to evacuate hematomas, relieve pressure, and remove any foreign bodies.
5. Management of seizures and other neurological complications.
6. Close monitoring of vital signs, particularly blood pressure and oxygenation.
7. Supportive care, such as mechanical ventilation, to maintain adequate respiratory and cardiovascular function.
8. Rehabilitation therapies, including physical, occupational, and speech therapy, to address cognitive, emotional, and physical deficits.
9. Medications to manage pain, seizures, and other n

In [138]:
response_4c = generate_rag_response(query_4, vector_databases[1], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4c)


For a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, the recommended treatments include:

1. A team approach that combines physical, occupational, and speech therapy to build skills and address cognitive, emotional, and social needs.
2. Rehabilitation services that are tailored to the individual's specific needs and may include cognitive training, memory exercises, and behavioral interventions.
3. Counseling and support for the patient and their family to address any emotional or social challenges that may arise during the recovery process.
4. A rehabilitation plan that is developed early on in the recovery process and is regularly updated to reflect the patient's progress and changing needs.
5. Monitoring and tracking of the patient's condition, including regular assessments of cognitive, emotional, and social functioning, as well as any complications or side effects of treatment.
6. A multidisciplinary ap

In [139]:
response_4d = generate_rag_response(query_4, vector_databases[1], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4d)


For a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, the recommended treatments include:

1. Multidisciplinary rehabilitation approach that combines physical, occupational, and speech therapy to build skills and address cognitive, emotional, and social needs.
2. Early intervention and assessment by a team of specialists, including neurologists, neurosurgeons, rehabilitation physicians, and therapists, to evaluate the extent of the injury and develop an individualized treatment plan.
3. Monitoring and tracking of the patient's progress, with regular follow-up appointments to assess and adjust the treatment plan as needed.
4. Medications to manage seizures, edema, and other complications that may arise during the recovery process.
5. Surgical interventions, such as decompression or craniotomy, to relieve pressure on the brain and improve blood flow.
6. Rehabilitation services, including cognitive therapy, phy

In [140]:
response_4e = generate_rag_response(query_4, vector_databases[2], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4e)


For a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, the recommended treatments may include:

1. Surgical drainage of hematomas and cerebral edema.
2. Monitoring of vital signs, particularly airway, breathing, and blood pressure.
3. Management of intracranial pressure through elevation of the head, sedation, and analgesia.
4. Rehabilitation therapy, including physical, occupational, and speech therapy, to build skills and address cognitive, emotional, and social needs.
5. Counseling and support for the patient and their family, to address psychological and social issues.
6. Serial CT scans to monitor for any changes in the brain tissue and to adjust treatment accordingly.
7. Rehabilitation services, including physical, occupational, and speech therapy, to help regain lost functions and build new skills.
8. Cognitive and behavioral therapy to address any cognitive or emotional deficits.
9. Medications to man

In [141]:
response_4f = generate_rag_response(query_4, vector_databases[2], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4f)


Treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, include:

1. Surgical drainage of hematomas and cerebral edema.
2. Management of intracranial pressure and prevention of further brain damage.
3. Rehabilitation therapies such as physical, occupational, and speech therapy to build skills and address cognitive, emotional, and social needs.
4. Medications to manage seizures, pain, and other complications.
5. Close monitoring and serial CT scans to track progress and detect any complications or changes in the brain.
6. Early intervention and rehabilitation to maximize recovery and minimize long-term disability.
7. A team approach involving a multidisciplinary care team, including physicians, therapists, counselors, and social workers, to provide comprehensive care and support.


In [142]:
response_4g = generate_rag_response(query_4, vector_databases[3], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_4g)

What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
The recommended treatments for a person with a physical injury to brain tissue and subsequent impairment of brain function include:

1. Airway management and oxygenation to ensure adequate blood flow and prevent further damage.
2. Control of intracranial pressure through the use of medications or surgical interventions, as necessary.
3. Monitoring of vital signs, particularly blood pressure and temperature, to detect any changes or complications.
4. Imaging studies, such as CT scans, to assess the extent of the injury and identify any structural damage or hematomas.
5. Surgical interventions to relieve pressure, remove hematomas, or repair damaged blood vessels, as needed.
6. Rehabilitation therapies, including physical, occupational, and speech therapy, to help regain lost function and improve quality of life.
7. Medication

Response 4f is best in my opinion due to being specific on the required medical procedures, but others are still close and have their advantages and disadvantages

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [143]:
response_5a = generate_rag_response(query_5, vector_databases[0], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5a)


When a person fractures their leg during a hiking trip, it is essential to take necessary precautions and provide appropriate treatment steps for their care and recovery. The following are some crucial considerations:

1. Immobilization: Immediately immobilize the affected leg with a splint or cast to prevent further damage and promote healing.
2. Pain management: Administer pain medication as prescribed by a healthcare professional to manage discomfort and facilitate mobility.
3. Wound care: Keep the wound clean and dry, and monitor for signs of infection, such as redness, swelling, or increased pain.
4. Monitoring for complications: Watch for potential complications like deep vein thrombosis (DVT), pulmonary embolism (PE), or infection, and seek medical attention if any of these occur.
5. Rehabilitation: Begin rehabilitation as soon as possible to regain strength, mobility, and function in the affected leg. This may involve physical therapy, exercises, and assistive devices like wal

In [144]:
response_5b = generate_rag_response(query_5, vector_databases[0], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5b)


When a person fractures their leg during a hiking trip, it is essential to provide immediate first aid and seek medical attention as soon as possible. The necessary precautions and treatment steps include:

1. Immobilization: Immobilize the affected limb using a splint or cast to prevent further damage and reduce pain.
2. Pain management: Administer pain medication to alleviate discomfort and facilitate mobility.
3. Wound care: Clean and dress any wounds or lacerations to prevent infection and promote healing.
4. Monitoring: Closely monitor the patient's vital signs, particularly their heart rate and blood pressure, to ensure they are stable.
5. Transportation: Transport the patient to a medical facility as soon as possible, ideally by a stretcher or wheelchair, to receive further treatment and care.
6. Rehabilitation: Once the patient is stabilized, initiate rehabilitation exercises to improve strength, flexibility, and range of motion in the affected limb.
7. Prevention of complicat

In [145]:
response_5c = generate_rag_response(query_5, vector_databases[1], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5c)


When a person fractures their leg during a hiking trip, it is essential to take necessary precautions and treatment steps to ensure proper care and recovery. Here are some key considerations:

1. Rest, Ice, Compression, and Elevation (RICE): Apply RICE principles to reduce pain and swelling. Use ice packs for 15-20 minutes every hour, compress the affected area with an elastic bandage, and elevate it above heart level.
2. Weight-bearing status: Avoid putting weight on the affected leg as much as possible. If the fracture is displaced or open, use a cast or splint to immobilize the leg and prevent further damage.
3. Pain management: Use nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen or naproxen to manage pain. However, avoid overusing these medications, as they can cause gastrointestinal issues and renal damage.
4. Physical therapy: Once the pain subsides, initiate physical therapy exercises to strengthen the affected muscles and improve range of motion. Focus on exerc

In [146]:
response_5d = generate_rag_response(query_5, vector_databases[1], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5d)


For a person who has fractured their leg during a hiking trip, necessary precautions and treatment steps include:

1. Immobilization of the affected limb to prevent further injury and promote healing.
2. Pain management with non-steroidal anti-inflammatory drugs (NSAIDs) or other medications as needed.
3. Rest and limited weight-bearing activities for 6-12 weeks to allow the fracture to heal properly.
4. Physical therapy exercises to maintain strength and range of motion in the unaffected limb, and to prevent muscle atrophy and weakness.
5. Monitoring for signs of complications such as infection, nerve damage, or poor healing, and seeking medical attention if any of these occur.
6. Using supportive devices such as crutches, canes, or braces to assist with mobility and balance during the recovery period.
7. Avoiding activities that may exacerbate the injury, such as running or jumping, and taking regular breaks to rest the affected limb.
8. Keeping the affected limb elevated to reduce 

In [147]:
response_5e = generate_rag_response(query_5, vector_databases[2], k=3, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5e)


For a person who has fractured their leg during a hiking trip, necessary precautions and treatment steps include:

1. Immobilization of the affected limb to prevent further damage and promote healing.
2. Pain management with medication to alleviate discomfort and allow for proper immobilization.
3. Wound care to clean and dress the wound, and prevent infection.
4. Monitoring of vital signs, particularly heart rate and blood pressure, to ensure that there are no complications.
5. Close supervision by a physician or therapist to assess progress and adjust treatment as needed.
6. Use of canes or walkers for support during the recovery process.
7. Gradual progression of weight-bearing activities to allow the fracture to heal properly.
8. Monitoring for signs of infection, such as redness, swelling, or increased pain, and seeking medical attention if any are observed.
9. Consideration of physical therapy to help regain strength and mobility in the affected limb.
10. Education on prevention

In [148]:
response_5f = generate_rag_response(query_5, vector_databases[2], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5f)


For a person who has fractured their leg during a hiking trip, necessary precautions and treatment steps include:

1. Immobilization of the affected limb to prevent further injury and promote healing.
2. Pain management with medication to minimize discomfort and facilitate mobility.
3. Wound care to prevent infection and promote healing, including cleaning and dressing the wound, and monitoring for signs of infection.
4. Monitoring of vital signs, particularly heart rate and blood pressure, to ensure that there are no complications related to the injury or treatment.
5. Close supervision by a physician or therapist to assess progress and adjust treatment as needed.
6. Use of assistive devices such as canes or walkers to facilitate mobility and prevent falls.
7. Gradual progression of weight-bearing activities to allow the fracture to heal properly.
8. Monitoring for signs of neurotoxicity or envenomation, particularly in cases of snake bites, and administration of appropriate antiveno

In [149]:
response_5g = generate_rag_response(query_5, vector_databases[3], k=4, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(response_5g)


For a person who has fractured their leg during a hiking trip, necessary precautions and treatment steps include:

1. Immobilization of the affected limb to prevent further damage and promote healing.
2. Pain management with medication and/or alternative therapies such as physical therapy or acupuncture.
3. Wound care and infection prevention, including cleaning and dressing the wound, and monitoring for signs of infection.
4. Monitoring for neurological and circulatory complications, such as nerve damage or poor blood flow.
5. Early mobilization and rehabilitation to prevent joint stiffness and promote recovery.
6. Proper fitting of prosthetic devices, if applicable.
7. Regular follow-up appointments with a healthcare provider to monitor progress and address any complications.

It is important to consider the patient's individual needs and preferences when developing their care and recovery plan, including their age, overall health, and activity level. Additionally, it may be necessa

All the responses here are similarly good, but due to the success of the f combination in other query responses, I will use that going forward

## Output Evaluation

In [150]:
chosen_vector_db = vector_databases[2]
chosen_k=4

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

**Note that here, I am choosing to use a chosen vector_db as vector_databases[2] and chosen_k as 4 to represent a single RAG system as my final RAG system, rather than evaluating multiple RAG systems at once**

In [151]:
groundedness_rater_system_message  = """
You will be presented a ###Question, ###Context used by the AI system and AI generated ###Answer.

Your task is to judge the extent to which the ###Answer is derived from ###Context.

Rate it 1 - if The ###Answer is not derived from the ###Context at all
Rate it 2 - if The ###Answer is derived from the ###Context only to a limited extent
Rate it 3 - if The ###Answer is derived from ###Context to a good extent
Rate it 4 - if The ###Answer is derived from ###Context mostly
Rate it 5 - if The ###Answer is is derived from ###Context completely

The final output should be a single overall rating in the range of 1 to 5, along with a brief explanation of the rationale for the rating.
"""

In [152]:
relevance_rater_system_message = """
You will be presented with a ###Question, the ###Context used by the AI system to generate a response, and the AI-generated ###Answer.

Your task is to judge the extent to which the ###Answer is relevant to the ###Question, considering whether it directly addresses the key aspects of the ###Question based on the provided ###Context.

Rate the relevance as follows:
- Rate 1 – The ###Answer is not relevant to the ###Question at all.
- Rate 2 – The ###Answer is only slightly relevant to the **###Question**, missing key aspects.
- Rate 3 – The ###Answer is moderately relevant, addressing some parts of the **###Question** but leaving out important details.
- Rate 4 – The ###Answer is mostly relevant, covering key aspects but with minor gaps.
- Rate 5 – The ###Answer is fully relevant, directly answering all important aspects of the **###Question** with appropriate details from the **###Context**.

The final output should be a single overall rating in the range of 1 to 5, along with a brief explanation of the rationale for the rating.
"""

In [153]:
def generate_ground_relevance_response(user_input, vector_db, k=3,max_tokens=128,temperature=0,top_p=0.9,top_k=35):
    global groundedness_rater_system_message, relevance_rater_system_message, user_message_template # user_message_template is for evaluation

    # Retrieve relevant document chunks
    retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": k})
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Define a simple generation prompt for the answer itself, without the '###Answer' tag
    # This prevents the LLM from generating '###References' prematurely.
    answer_generation_prompt = f"""
###Question
{user_input}

###Context
{context_for_query}

Based on the provided context, please answer the question.
Answer:
"""

    # Generate the initial answer using the simple generation prompt
    # The previous `llm` calls were directly `llm(prompt=...)` and returned a string if verbose=False,
    # or a dictionary if verbose=True. Let's ensure compatibility.
    generated_answer_raw = llm(
            prompt=answer_generation_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['\n###', '###', 'Question:', 'Context:', 'Answer:', 'INST'], # Assuming 'INST' might be a stop token if it's llama-based prompting
            )

    # Extract the actual answer text. If `llm` returns a dict (e.g., from llama_cpp.Llama),
    # we need to extract from `choices`.
    answer = ''
    if isinstance(generated_answer_raw, dict) and 'choices' in generated_answer_raw and len(generated_answer_raw['choices']) > 0:
        answer = generated_answer_raw['choices'][0]['text']
    else:
        answer = str(generated_answer_raw)


    # Now, use the evaluation templates for groundedness and relevance ratings
    # These templates expect 'answer' to be filled in.

    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    relevance_prompt = f"""[INST]{relevance_rater_system_message}
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Generate groundedness rating
    response_1_raw = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    # Generate relevance rating
    response_2_raw = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    # Extract text from raw responses
    response_1_text = ''
    if isinstance(response_1_raw, dict) and 'choices' in response_1_raw and len(response_1_raw['choices']) > 0:
        response_1_text = response_1_raw['choices'][0]['text']
    else:
        response_1_text = str(response_1_raw)

    response_2_text = ''
    if isinstance(response_2_raw, dict) and 'choices' in response_2_raw and len(response_2_raw['choices']) > 0:
        response_2_text = response_2_raw['choices'][0]['text']
    else:
        response_2_text = str(response_2_raw)

    return response_1_text, response_2_text


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [154]:
ground_relevance_1 = generate_ground_relevance_response(query_1, chosen_vector_db, k=chosen_k, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(ground_relevance_1)

('  Based on the provided information, I would rate the answer as a 4 out of 5 in terms of how well it is derived from the context. The answer provides a comprehensive list of protocols for managing sepsis in a critical care unit, including early recognition and diagnosis, aggressive fluid resuscitation, administration of broad-spectrum antibiotics, surgical intervention, close monitoring of vital signs, regular assessment of organ function, management of hypotension and shock, protocol-driven management, frequent blood cultures, and early recognition and management of complications.\n\nThe answer is well-organized and includes all the important aspects of sepsis management. However, to reach a perfect 5 out of 5, the answer could benefit from more specific details on how these protocols are implemented in practice, such as the specific antibiotics and vasopressors used, the target blood pressure and heart rate ranges, and the criteria for escalating treatment to more intensive care. A

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [155]:
ground_relevance_2 = generate_ground_relevance_response(query_2, chosen_vector_db, k=chosen_k, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(ground_relevance_2)

('  Based on the provided information, I would rate the answer as a 4 out of 5 in terms of how well it is derived from the context. The answer provides a comprehensive list of common symptoms for appendicitis and describes the appropriate surgical procedure for treating the condition. The answer also highlights the importance of seeking medical attention immediately if symptoms of appendicitis are present, which is consistent with the context provided.\n\nThe only aspect where the answer could be improved is in providing more specific information about the etiology of appendicitis and the potential complications that can arise if it is not treated promptly. For example, the answer could mention that appendicitis is caused by an obstruction of the appendix, which can lead to inflammation and infection, and that delaying treatment can increase the risk of peritonitis or death. Additionally, the answer could provide more information about the laparoscopic appendectomy procedure, such as t

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [156]:
ground_relevance_3 = generate_ground_relevance_response(query_3, chosen_vector_db, k=chosen_k, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(ground_relevance_3)

('  Based on the provided context and answer, I would rate the answer as a 4 out of 5 in terms of how well it is derived from the context. The answer provides a comprehensive list of effective treatments and solutions for addressing sudden patchy hair loss, and each treatment option is supported by evidence from the provided context. However, the answer could be improved by providing more specific examples or case studies to further illustrate the effectiveness of each treatment option.\n\nThe answer effectively covers all the key points mentioned in the context, including the use of topical corticosteroids, anthralin, minoxidil, oral antifungals, immunotherapy, surgical options, hormone modulators, and psychological interventions. The answer also acknowledges the importance of a thorough evaluation by a dermatologist to determine the most appropriate treatment approach.\n\nOverall, the answer is well-structured and provides a clear overview of the various treatment options available f

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [157]:
ground_relevance_4 = generate_ground_relevance_response(query_4, chosen_vector_db, k=chosen_k, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(ground_relevance_4)

('  Based on the provided information, I would rate the answer as a 4, meaning that the answer is derived from the context to a good extent. The answer provides a comprehensive overview of the treatments that may be recommended for a person with a physical injury to brain tissue, including surgical interventions and rehabilitation therapies. The answer also mentions the importance of early intervention and a team approach to care, which is consistent with the information provided in the context.\n\nHowever, there are a few areas where the answer could be improved. Firstly, the answer does not provide any specific examples or case studies to illustrate the treatments being discussed. Additionally, the answer does not fully address the issue of temporary vs permanent impairment of brain function, which is mentioned in the context. To address these limitations, I would suggest providing more specific examples and case studies to illustrate the treatments being recommended, as well as disc

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [158]:
ground_relevance_5 = generate_ground_relevance_response(query_5, chosen_vector_db, k=chosen_k, max_tokens=1024, temperature=0, top_p=0.9, top_k=35)
print(ground_relevance_5)

('  Based on the provided context and answer, I would rate the answer as a 4 out of 5 in terms of how well it is derived from the context. The answer provides a comprehensive list of necessary precautions and treatment steps for a person with a fractured leg who has experienced a fall during a hiking trip, and it is largely consistent with the information provided in the context.\n\nThe answer does an excellent job of highlighting the importance of immobilizing the affected limb, assessing for other injuries, monitoring vital signs, providing pain management, preventing infection, and transporting the patient to a medical facility. Additionally, it mentions the possibility of envenomation from snake bites and the need for ongoing care upon arrival at the hospital.\n\nHowever, there are a few minor areas where the answer could be improved. For example, the answer could have provided more specific information about how to immobilize the affected limb (e.g., using a splint or cast) and ho

## Actionable Insights and Business Recommendations

This model is great for people who are not medical providers to help get an overview of preventative measures and initial steps to take in certain medical events, but it sometimes misses key details which might not be okay to miss for a doctor who is actually treating a pateint. However, even for medical providers, this still provides most of the necessary details and can help simplify the process of brushing up on details versus reading through the whole manual; it's just that I don't think medical providers should become reliant on this model for everything

One thing to improve in the future is to remove all the references to outside material that is present in our data, as this is useless and in fact contaminating data for our model (which doesn't have information on these references). It's just that it would take more work to find where references to outside material are present and remove those parts of the text manually.

I could also test with more combinations of hyperparameters such as different values of top_p and top_k in the RAG fine-tuning part, but I couldn't do this due to time constraints.

<font size=6 color='blue'>Power Ahead</font>
___